In [ ]:
!pip install pandas requests tqdm

In [ ]:
import json
import time
import pandas as pd
from google.colab import userdata
from openai import OpenAI
from tqdm import tqdm

# 1. Fetch OpenRouter API key from Colab Secrets
try:
  api_key = userdata.get('OPENROUTER_API_KEY')
except Exception as e:
  raise ValueError(
      "Key 'OPENROUTER_API_KEY' not found in Colab Secrets."
  ) from e

client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=api_key)

# Model configured for generation task
GEN_MODEL = 'gpt-5.6-luna'

SYSTEM_PROMPT = """TASK
For each medical MCQ, rewrite the QUESTION using exactly ONE of the
three techniques below so none of options A–D remains a valid answer —
option E ("উত্তর নেই") becomes the only correct choice. Keep it fluent,
professional-sounding, natural Bengali.

TECHNIQUES
1) Contradiction — embed two medically incompatible premises that are
individually plausible but jointly inconsistent.
e.g., "যে অন্তঃক্ষরা গ্রন্থি ইনসুলিন ক্ষরণ করে, সেই গ্রন্থিটি বৃক্কের ঠিক উপরে
অবস্থিত — এর নাম কী?" (falsely gives the pancreas the adrenal gland's
location)

2) Unrealism — state a small, clinically-styled but factually wrong
value/range or property (not an absurd one), so catching it requires
knowing the real clinical fact.
e.g., "স্বাভাবিক উপবাসকালীন রক্তে গ্লুকোজের মাত্রা ১১০-১২৫ mg/dL ধরে নিলে,
অগ্ন্যাশয়ের কোন কোষ এই মাত্রা নিয়ন্ত্রণে সর্বাধিক ভূমিকা রাখে?" (this range
is actually prediabetic, not normal; real normal is 70–99 mg/dL)
e.g., "অ্যান্টিবডি উৎপাদনে সক্রিয় টি-লিম্ফোসাইট কোষ বিবেচনায় নিলে, অস্থিমজ্জার
কোন অংশ এই কোষ উৎপাদনে সর্বাধিক ভূমিকা রাখে?" (antibodies are actually
produced by B-lymphocytes/plasma cells, not T-lymphocytes — a subtle,
plausible-sounding swap rather than a numeric one)

3) Referential Deletion — delete the essential piece of information the
question depends on, disguised as a confident anaphoric reference
("উক্ত," "উল্লিখিত," "এই নির্দিষ্ট") whose antecedent was never given.
e.g., "উল্লিখিত হরমোনের ঘাটতিতে মানবদেহে কোন লক্ষণ সবচেয়ে আগে দেখা দেয়?"
(no hormone was ever named, so the answer depends on unstated info)

RULES
1) A STRONG modified question reads exactly like a normal question and
   requires correct medical knowledge to detect the flaw.
2) The paired options A-D must no longer be valid/correct answers.
3) Keep the modified question in fluent, natural Bengali.
4) Distribute techniques across the dataset based on best fit.
5) Provide a 1-sentence rationale explaining why a domain expert would catch the issue.

OUTPUT FORMAT (JSON Object):
{
  "modified_question": "<rewritten Bengali question>",
  "technique": "<Contradiction | Unrealism | Referent Deletion>",
  "difficulty_justification": "<1 sentence rationale>"
}"""


def call_generation_llm(user_content):
  response = client.chat.completions.create(
      model=GEN_MODEL,
      messages=[
          {'role': 'system', 'content': SYSTEM_PROMPT},
          {'role': 'user', 'content': user_content},
      ],
      response_format={'type': 'json_object'},
      temperature=0.3,
  )
  return json.loads(response.choices[0].message.content)


# Load baseline dataset
df = pd.read_csv('bangla_med_qa_5options_clean.csv')
OUTPUT_FILE = 'bangla_med_qa_hallucinated.csv'

# Initialize tracking columns while PRESERVING original 'question'
if 'hallucinated_question' not in df.columns:
  df['hallucinated_question'] = None
if 'technique' not in df.columns:
  df['technique'] = None
if 'difficulty_justification' not in df.columns:
  df['difficulty_justification'] = None

print('Starting hallucination generation process...')

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Generating Flaws'):
  # Skip already processed rows to support resumption
  if (
      pd.notna(df.at[idx, 'hallucinated_question'])
      and str(df.at[idx, 'hallucinated_question']).strip() != ''
  ):
    continue

  user_payload = (
      f"Original Question: {row['question']}\n"
      f'Options:\n'
      f"A: {row['options/A']}\n"
      f"B: {row['options/B']}\n"
      f"C: {row['options/C']}\n"
      f"D: {row['options/D']}\n"
      f"Current Correct Answer: {row['answer']}"
  )

  retries = 3
  for attempt in range(retries):
    try:
      result = call_generation_llm(user_payload)

      # Keep original 'question' intact; store flawed version in 'hallucinated_question'
      df.at[idx, 'hallucinated_question'] = result.get('modified_question')
      df.at[idx, 'technique'] = result.get('technique')
      df.at[idx, 'difficulty_justification'] = result.get(
          'difficulty_justification'
      )
      break
    except Exception as e:
      if attempt == retries - 1:
        print(f'\n[Warning] Row {idx} failed: {e}')
      else:
        time.sleep(2)

  if idx % 10 == 0:
    df.to_csv(OUTPUT_FILE, index=False)

df.to_csv(OUTPUT_FILE, index=False)
print(f'\nGeneration complete! Output saved to: {OUTPUT_FILE}')

Starting hallucination generation process...


Generating Flaws: 100%|██████████| 994/994 [1:20:18<00:00,  4.85s/it]


Generation complete! Output saved to: bangla_med_qa_hallucinated.csv


In [ ]:
import json
import re
import time
import pandas as pd
from google.colab import userdata
from openai import OpenAI
from tqdm import tqdm

try:
  api_key = userdata.get('OPENROUTER_API_KEY')
except Exception as e:
  raise ValueError(
      "Key 'OPENROUTER_API_KEY' not found in Colab Secrets."
  ) from e

client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=api_key)

# 6 Target evaluation models mapped to OpenRouter endpoints
EVAL_MODELS = {
    'GPT-4o-mini': 'openai/gpt-4o-mini',
    'Gemini-3.1-flash-lite': 'google/gemini-3.1-flash-lite',
    'Grok-4.3': 'x-ai/grok-4.3',
    'Qwen-2.5-72B': 'qwen/qwen-2.5-72b-instruct',
    'Llama-3.1-70B': 'meta-llama/llama-3.1-70b-instruct',
    'DeepSeek-V4-Flash': 'deepseek/deepseek-v4-flash',
}

EVAL_SYSTEM_PROMPT = """You are an expert physician evaluating a multiple-choice question.
Read the question and options carefully. Identify any factual errors, contradictory premises, or invalid clinical thresholds.

If NONE of options A, B, C, or D is medically correct due to a flawed or hallucinated question premise, select option E ("উত্তর নেই").

Respond ONLY in valid JSON format:
{
  "selected_option": "<A | B | C | D | E>",
  "explanation": "<1 short sentence rationale>"
}"""


def evaluate_mcq(model_slug, question, opt_a, opt_b, opt_c, opt_d, opt_e):
  user_prompt = (
      f'Question: {question}\n'
      f'Options:\n'
      f'A: {opt_a}\n'
      f'B: {opt_b}\n'
      f'C: {opt_c}\n'
      f'D: {opt_d}\n'
      f'E: {opt_e}\n\n'
      f'Which option (A, B, C, D, or E) is strictly correct?'
  )

  retries = 3
  for attempt in range(retries):
    try:
      response = client.chat.completions.create(
          model=model_slug,
          messages=[
              {'role': 'system', 'content': EVAL_SYSTEM_PROMPT},
              {'role': 'user', 'content': user_prompt},
          ],
          response_format={'type': 'json_object'},
          temperature=0.0,
      )
      parsed = json.loads(response.choices[0].message.content)
      choice = str(parsed.get('selected_option', '')).strip().upper()

      match = re.search(r'[A-E]', choice)
      return match.group(0) if match else 'UNKNOWN'
    except Exception:
      if attempt == retries - 1:
        return 'ERROR'
      time.sleep(2)


# Load dataset containing hallucinated questions
try:
  df_eval = pd.read_csv('bangla_med_qa_hallucinated_100.csv')
except FileNotFoundError:
  df_eval = pd.read_csv('bangla_med_qa_hallucinated.csv')

EVAL_OUTPUT_FILE = 'bangla_med_qa_model_evaluations.csv'
print(f'Loaded {len(df_eval)} questions for benchmark evaluation.')

# Run evaluation across all 6 models
for model_name, model_slug in EVAL_MODELS.items():
  pred_col = f'pred_{model_name}'
  corr_col = f'correct_{model_name}'

  if pred_col not in df_eval.columns:
    df_eval[pred_col] = None
    df_eval[corr_col] = None

  print(f'\nRunning Evaluation: {model_name} ({model_slug})')

  for idx, row in tqdm(
      df_eval.iterrows(), total=len(df_eval), desc=f'Testing {model_name}'
  ):
    # Resume safeguard
    if pd.notna(df_eval.at[idx, pred_col]) and str(
        df_eval.at[idx, pred_col]
    ) not in ['', 'ERROR']:
      continue

    # Use hallucinated_question; fallback to question if missing
    eval_q = (
        row['hallucinated_question']
        if pd.notna(row.get('hallucinated_question'))
        else row['question']
    )
    opt_e_val = row['options/E'] if 'options/E' in row else 'উত্তর নেই'

    pred = evaluate_mcq(
        model_slug=model_slug,
        question=eval_q,
        opt_a=row['options/A'],
        opt_b=row['options/B'],
        opt_c=row['options/C'],
        opt_d=row['options/D'],
        opt_e=opt_e_val,
    )

    df_eval.at[idx, pred_col] = pred
    # Option 'E' indicates successful detection of the hallucinated premise
    df_eval.at[idx, corr_col] = 1 if pred == 'E' else 0

    if idx % 10 == 0:
      df_eval.to_csv(EVAL_OUTPUT_FILE, index=False)

  df_eval.to_csv(EVAL_OUTPUT_FILE, index=False)

print(
    f'\nAll evaluations completed! Full results stored in: {EVAL_OUTPUT_FILE}'
)

Loaded 994 questions for benchmark evaluation.

Running Evaluation: GPT-4o-mini (openai/gpt-4o-mini)


Testing GPT-4o-mini: 100%|██████████| 994/994 [21:03<00:00,  1.27s/it]



Running Evaluation: Gemini-3.1-flash-lite (google/gemini-3.1-flash-lite)


Testing Gemini-3.1-flash-lite: 100%|██████████| 994/994 [14:38<00:00,  1.13it/s]



Running Evaluation: Grok-4.3 (x-ai/grok-4.3)


Testing Grok-4.3: 100%|██████████| 994/994 [1:14:11<00:00,  4.48s/it]



Running Evaluation: Qwen-2.5-72B (qwen/qwen-2.5-72b-instruct)


Testing Qwen-2.5-72B: 100%|██████████| 994/994 [1:08:35<00:00,  4.14s/it]



Running Evaluation: Llama-3.1-70B (meta-llama/llama-3.1-70b-instruct)


Testing Llama-3.1-70B:  56%|█████▋    | 561/994 [2:51:39<09:47,  1.36s/it]

In [ ]:
import pandas as pd

df_results = pd.read_csv('bangla_med_qa_model_evaluations.csv')
summary_data = []

for model_name in EVAL_MODELS.keys():
  pred_col = f'pred_{model_name}'
  corr_col = f'correct_{model_name}'

  if corr_col in df_results.columns:
    total_eval = df_results[corr_col].notna().sum()
    total_correct = df_results[corr_col].sum()
    accuracy = (total_correct / total_eval * 100) if total_eval > 0 else 0.0

    counts = df_results[pred_col].value_counts().to_dict()
    chose_e = counts.get('E', 0)
    failed_a_d = sum(counts.get(k, 0) for k in ['A', 'B', 'C', 'D'])

    summary_data.append({
        'Model': model_name,
        'Evaluated': total_eval,
        'Detected Flaw (Selected E)': int(chose_e),
        'Hallucinated Answer (Selected A-D)': int(failed_a_d),
        'Accuracy Rate (%)': round(accuracy, 2),
    })

summary_df = pd.DataFrame(summary_data).sort_values(
    by='Accuracy Rate (%)', ascending=False
)

print('=== Hallucination Resistance Benchmark Summary ===')
print(summary_df.to_string(index=False))